In [ ]:
import numpy as np
import pandas as pd
import os

# --- CONFIG ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"
MERGED_CSV_PATH = os.path.join(BASE_PATH, "merged_components_consistent.csv")
RAW_PAYLOAD_PATH = os.path.join(BASE_PATH, "cnn_payload_data.npy")

# OUTPUTS
OUT_VIEW1 = os.path.join(BASE_PATH, "v3PRETRAIN_X_view1.npy")
OUT_VIEW2 = os.path.join(BASE_PATH, "v3PRETRAIN_X_view2.npy")

def main():
    print("--- Regenerating Clean Pre-training Data ---")

    # 1. Load & Clean View 2 (Statistics)
    if not os.path.exists(MERGED_CSV_PATH):
        print(f"Error: CSV not found at {MERGED_CSV_PATH}")
        return

    print(f"Loading CSV: {MERGED_CSV_PATH}")
    df = pd.read_csv(MERGED_CSV_PATH)

    # --- CRITICAL FIX: Force Numeric Conversion ---
    # This turns any text columns (like 'filename') into NaN, then we drop them.
    print("Forcing numeric conversion on View 2...")

    # Identify columns to explicit exclude (Labels)
    exclude_cols = ['filename', 'application', 'category', 'binary_type']

    # Drop known text columns first
    df_clean = df.drop(columns=[c for c in exclude_cols if c in df.columns], errors='ignore')

    # Also drop any Alpha embeddings if they exist in CSV (we want Raw Payload instead)
    df_clean = df_clean.drop(columns=[c for c in df_clean.columns if c.startswith('alpha_')], errors='ignore')

    # Force everything else to numeric (coerce errors to NaN)
    df_numeric = df_clean.apply(pd.to_numeric, errors='coerce')

    # Drop columns that became all NaN (meaning they were text)
    df_numeric = df_numeric.dropna(axis=1, how='all')

    # Fill remaining NaNs with 0
    df_numeric = df_numeric.fillna(0)

    X_view2 = df_numeric.values.astype('float32')
    print(f"Cleaned View 2 Shape: {X_view2.shape} (Must be purely numeric)")

    # 2. Load View 1 (Raw Payload)
    print("Loading Raw Payload...")
    if not os.path.exists(RAW_PAYLOAD_PATH):
        print(f"Error: Raw payload not found at {RAW_PAYLOAD_PATH}")
        return

    # Load raw bytes
    X_view1_all = np.load(RAW_PAYLOAD_PATH)

    # Align rows (CSV rows must match Payload rows)
    # We assume the CSV was generated from the Payload file and shares the same index order.
    # If lengths differ, we truncate to the smaller size (safe fallback).
    min_len = min(len(X_view2), len(X_view1_all))
    X_view1 = X_view1_all[:min_len].astype('float32')
    X_view2 = X_view2[:min_len]

    print(f"Aligned View 1 Shape: {X_view1.shape}")
    print(f"Aligned View 2 Shape: {X_view2.shape}")

    # 3. Save Cleaned Data
    np.save(OUT_VIEW1, X_view1)
    np.save(OUT_VIEW2, X_view2)
    print("Success! Clean, numeric .npy files saved.")
    print("You can now run the Training or t-SNE script.")

if __name__ == "__main__":
    main()

In [4]:
import numpy as np
import pandas as pd
import os

# --- CONFIG ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"
MERGED_CSV_PATH = os.path.join(BASE_PATH, "VNAT_merged_components_consistent.csv")
RAW_PAYLOAD_PATH = os.path.join(BASE_PATH, "VNAT_cnn_payload_data.npy")

# OUTPUTS
OUT_VIEW1 = os.path.join(BASE_PATH, "VNAT_v3PRETRAIN_X_view1.npy")
OUT_VIEW2 = os.path.join(BASE_PATH, "VNAT_v3PRETRAIN_X_view2.npy")

def main():
    print("--- Regenerating Clean Pre-training Data ---")

    # 1. Load & Clean View 2 (Statistics)
    if not os.path.exists(MERGED_CSV_PATH):
        print(f"Error: CSV not found at {MERGED_CSV_PATH}")
        return

    print(f"Loading CSV: {MERGED_CSV_PATH}")
    df = pd.read_csv(MERGED_CSV_PATH)

    # --- CRITICAL FIX: Force Numeric Conversion ---
    # This turns any text columns (like 'filename') into NaN, then we drop them.
    print("Forcing numeric conversion on View 2...")

    # Identify columns to explicit exclude (Labels)
    exclude_cols = ['filename', 'application', 'category', 'binary_type']

    # Drop known text columns first
    df_clean = df.drop(columns=[c for c in exclude_cols if c in df.columns], errors='ignore')

    # Also drop any Alpha embeddings if they exist in CSV (we want Raw Payload instead)
    df_clean = df_clean.drop(columns=[c for c in df_clean.columns if c.startswith('alpha_')], errors='ignore')

    # Force everything else to numeric (coerce errors to NaN)
    df_numeric = df_clean.apply(pd.to_numeric, errors='coerce')

    # Drop columns that became all NaN (meaning they were text)
    df_numeric = df_numeric.dropna(axis=1, how='all')

    # Fill remaining NaNs with 0
    df_numeric = df_numeric.fillna(0)

    X_view2 = df_numeric.values.astype('float32')
    print(f"Cleaned View 2 Shape: {X_view2.shape} (Must be purely numeric)")

    # 2. Load View 1 (Raw Payload)
    print("Loading Raw Payload...")
    if not os.path.exists(RAW_PAYLOAD_PATH):
        print(f"Error: Raw payload not found at {RAW_PAYLOAD_PATH}")
        return

    # Load raw bytes
    X_view1_all = np.load(RAW_PAYLOAD_PATH)

    # Align rows (CSV rows must match Payload rows)
    # We assume the CSV was generated from the Payload file and shares the same index order.
    # If lengths differ, we truncate to the smaller size (safe fallback).
    min_len = min(len(X_view2), len(X_view1_all))
    X_view1 = X_view1_all[:min_len].astype('float32')
    X_view2 = X_view2[:min_len]

    print(f"Aligned View 1 Shape: {X_view1.shape}")
    print(f"Aligned View 2 Shape: {X_view2.shape}")

    # 3. Save Cleaned Data
    np.save(OUT_VIEW1, X_view1)
    np.save(OUT_VIEW2, X_view2)
    print("Success! Clean, numeric .npy files saved.")
    print("You can now run the Training or t-SNE script.")

if __name__ == "__main__":
    main()

--- Regenerating Clean Pre-training Data ---
Loading CSV: /content/drive/MyDrive/1 Skripsi/27jan/VNAT_merged_components_consistent.csv
Forcing numeric conversion on View 2...
Cleaned View 2 Shape: (3709, 137) (Must be purely numeric)
Loading Raw Payload...
Aligned View 1 Shape: (3709, 10, 784)
Aligned View 2 Shape: (3709, 137)
Success! Clean, numeric .npy files saved.
You can now run the Training or t-SNE script.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
